### Import modules and data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.data.cleaning import clean_string_series
from sklearn.metrics.pairwise import haversine_distances

In [2]:
detailed_path = PATHS['raw_data_notebooks'] / 'UTF8list-3.tsv'
detailed_reservoirs_pd = pd.read_csv(detailed_path, sep='\t')
detailed_reservoirs_pd.head()

,CODE,NAME,RESERVOIR,X,Y,BASIN,RIVERBED,GOOGLE,OPENSTREETMAP,WIKIDATA,PROVINCE,AUTONOMOUS_COMMUNITY,TYPE,CREST_ELEVATION,DAM_HEIGHT,REPORT
0,9250031,SALLENTE,SALLENTE,"42,5003709690001","0,991459934000034",EBRO,BARRANC DE LA LORA,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos de pantalla asfáltica,1770,89,https://sig.mapama.gob.es/WebServices/clientew...
1,3450047,CAZALEGAS,Cazalegas Dam,"40,01298709","-4,70624804799996",TAJO,RÍO ALBERCHE,https://www.google.com/search?kgmid=/g/11dfr4j268,NaN,https://www.wikidata.org/wiki/Q30278314,Toledo,Castilla - La Mancha,Presa de materiales sueltos de pantalla de hor...,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,9250014,LAGO NEGRO,LAGO NEGRO,"42,5422861970001","1,04058219400002",EBRO,NaN,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos zonificada o de nú...,2340,11,https://sig.mapama.gob.es/WebServices/clientew...
3,9500034,"TORCAS, LAS",Las Trocas reservoir,"41,294686266","-1,08744698599997",EBRO,RÍO HUERVA,NaN,NaN,https://www.wikidata.org/wiki/Q23991169,Zaragoza,Aragón,Presa de fábrica de gravedad (hormigón vibrado),"624,799999999999","39,45",https://sig.mapama.gob.es/WebServices/clientew...
4,3190006,"BUJEDA, LA","BUJEDA, LA","40,244500658","-2,83518801499997",TAJO,SIN NOMBRE,NaN,NaN,NaN,Guadalajara,Castilla - La Mancha,Presa de materiales sueltos homogénea,"905,5","39,5",https://sig.mapama.gob.es/WebServices/clientew...


In [3]:
detailed_reservoirs_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   CODE                  367 non-null    int64 
 1   NAME                  367 non-null    object
 2   RESERVOIR             367 non-null    object
 3   X                     367 non-null    object
 4   Y                     367 non-null    object
 5   BASIN                 367 non-null    object
 6   RIVERBED              359 non-null    object
 7   GOOGLE                115 non-null    object
 8   OPENSTREETMAP         36 non-null     object
 9   WIKIDATA              187 non-null    object
 10  PROVINCE              367 non-null    object
 11  AUTONOMOUS_COMMUNITY  367 non-null    object
 12  TYPE                  367 non-null    object
 13  CREST_ELEVATION       184 non-null    object
 14  DAM_HEIGHT            184 non-null    object
 15  REPORT                367 non-null    ob

### Renaming columns

In [4]:
# Will assign them to their lowercase version
detailed_reservoirs_pd.columns = detailed_reservoirs_pd.columns.str.lower()
columns_dict = {'x': 'longitude', 'y': 'latitude'}
detailed_reservoirs_pd.rename(columns=columns_dict, inplace=True)

### Looking for Missing Values before cleaning strings and converting dtypes

In [5]:
detailed_reservoirs_pd.isna().sum()

code                      0
name                      0
reservoir                 0
longitude                 0
latitude                  0
basin                     0
riverbed                  8
google                  252
openstreetmap           331
wikidata                180
province                  0
autonomous_community      0
type                      0
crest_elevation         183
dam_height              183
report                    0
dtype: int64

### Cleaning string data

- Without missing values:

In [6]:
def clean_string_columns(detailed_reservoirs_data, columns):
    for column in columns:
        detailed_reservoirs_data.loc[:, column] = clean_string_series(detailed_reservoirs_data[column])

clean_string_columns(detailed_reservoirs_pd, ['name', 'reservoir', 'basin', 'province', 'autonomous_community', 'type'])

- With missing values:

In [7]:
detailed_reservoirs_pd.loc[detailed_reservoirs_pd['riverbed'].notna(), 'riverbed'] = clean_string_series(detailed_reservoirs_pd.loc[detailed_reservoirs_pd['riverbed'].notna(), 'riverbed'])

In [8]:
detailed_reservoirs_pd.head()

,code,name,reservoir,longitude,latitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,9250031,sallente,sallente,"42,5003709690001","0,991459934000034",ebro,barranc lora,NaN,NaN,NaN,lleida,cataluna,presa materiales sueltos pantalla asfaltica,1770,89,https://sig.mapama.gob.es/WebServices/clientew...
1,3450047,cazalegas,cazalegas dam,"40,01298709","-4,70624804799996",tajo,rio alberche,https://www.google.com/search?kgmid=/g/11dfr4j268,NaN,https://www.wikidata.org/wiki/Q30278314,toledo,castilla mancha,presa materiales sueltos pantalla hormigon,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,9250014,lago negro,lago negro,"42,5422861970001","1,04058219400002",ebro,NaN,NaN,NaN,NaN,lleida,cataluna,presa materiales sueltos zonificada o nucleo,2340,11,https://sig.mapama.gob.es/WebServices/clientew...
3,9500034,torcas,trocas reservoir,"41,294686266","-1,08744698599997",ebro,rio huerva,NaN,NaN,https://www.wikidata.org/wiki/Q23991169,zaragoza,aragon,presa fabrica gravedad (hormigon vibrado),"624,799999999999","39,45",https://sig.mapama.gob.es/WebServices/clientew...
4,3190006,bujeda,bujeda,"40,244500658","-2,83518801499997",tajo,sin nombre,NaN,NaN,NaN,guadalajara,castilla mancha,presa materiales sueltos homogenea,"905,5","39,5",https://sig.mapama.gob.es/WebServices/clientew...


### Converting coordinate columns

In [9]:
detailed_reservoirs_pd['longitude'] = detailed_reservoirs_pd['longitude'].str.replace(',', '.').astype(float)
detailed_reservoirs_pd['latitude'] = detailed_reservoirs_pd['latitude'].str.replace(',', '.').astype(float)

### Converting other columns to float

In [10]:
detailed_reservoirs_pd.loc[detailed_reservoirs_pd['crest_elevation'].notna(), 'crest_elevation'] = detailed_reservoirs_pd.loc[detailed_reservoirs_pd['crest_elevation'].notna(), 'crest_elevation'].str.replace(',', '.').astype(float)
detailed_reservoirs_pd.loc[detailed_reservoirs_pd['dam_height'].notna(), 'dam_height'] = detailed_reservoirs_pd.loc[detailed_reservoirs_pd['dam_height'].notna(), 'dam_height'].str.replace(',', '.').astype(float)

In [11]:
detailed_reservoirs_pd.head()

,code,name,reservoir,longitude,latitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,9250031,sallente,sallente,42.500371,0.991460,ebro,barranc lora,NaN,NaN,NaN,lleida,cataluna,presa materiales sueltos pantalla asfaltica,1770.0,89.0,https://sig.mapama.gob.es/WebServices/clientew...
1,3450047,cazalegas,cazalegas dam,40.012987,-4.706248,tajo,rio alberche,https://www.google.com/search?kgmid=/g/11dfr4j268,NaN,https://www.wikidata.org/wiki/Q30278314,toledo,castilla mancha,presa materiales sueltos pantalla hormigon,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,9250014,lago negro,lago negro,42.542286,1.040582,ebro,NaN,NaN,NaN,NaN,lleida,cataluna,presa materiales sueltos zonificada o nucleo,2340.0,11.0,https://sig.mapama.gob.es/WebServices/clientew...
3,9500034,torcas,trocas reservoir,41.294686,-1.087447,ebro,rio huerva,NaN,NaN,https://www.wikidata.org/wiki/Q23991169,zaragoza,aragon,presa fabrica gravedad (hormigon vibrado),624.8,39.45,https://sig.mapama.gob.es/WebServices/clientew...
4,3190006,bujeda,bujeda,40.244501,-2.835188,tajo,sin nombre,NaN,NaN,NaN,guadalajara,castilla mancha,presa materiales sueltos homogenea,905.5,39.5,https://sig.mapama.gob.es/WebServices/clientew...


### Save current cleaning and developing detailed_reservoir.ipynb at EDA folder

In [12]:
cleaned_detailed_reservoir_path = PATHS['pre_EDA'] / 'detailed_reservoir_for_EDA.csv'
cleaned_detailed_reservoir_path.parent.mkdir(parents=True, exist_ok=True)
detailed_reservoirs_pd.to_csv(cleaned_detailed_reservoir_path, index=False)

## Post EDA Analysis

### Deleting Code and Reservoir Columns

In [13]:
detailed_reservoirs_pd = detailed_reservoirs_pd.drop(columns=['code', 'reservoir'])

### Handling Missing Values

In [14]:
detailed_reservoirs_pd.head()

,name,longitude,latitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,sallente,42.500371,0.991460,ebro,barranc lora,NaN,NaN,NaN,lleida,cataluna,presa materiales sueltos pantalla asfaltica,1770.0,89.0,https://sig.mapama.gob.es/WebServices/clientew...
1,cazalegas,40.012987,-4.706248,tajo,rio alberche,https://www.google.com/search?kgmid=/g/11dfr4j268,NaN,https://www.wikidata.org/wiki/Q30278314,toledo,castilla mancha,presa materiales sueltos pantalla hormigon,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,lago negro,42.542286,1.040582,ebro,NaN,NaN,NaN,NaN,lleida,cataluna,presa materiales sueltos zonificada o nucleo,2340.0,11.0,https://sig.mapama.gob.es/WebServices/clientew...
3,torcas,41.294686,-1.087447,ebro,rio huerva,NaN,NaN,https://www.wikidata.org/wiki/Q23991169,zaragoza,aragon,presa fabrica gravedad (hormigon vibrado),624.8,39.45,https://sig.mapama.gob.es/WebServices/clientew...
4,bujeda,40.244501,-2.835188,tajo,sin nombre,NaN,NaN,NaN,guadalajara,castilla mancha,presa materiales sueltos homogenea,905.5,39.5,https://sig.mapama.gob.es/WebServices/clientew...


In [15]:
detailed_reservoirs_pd.isna().sum()

name                      0
longitude                 0
latitude                  0
basin                     0
riverbed                  8
google                  252
openstreetmap           331
wikidata                180
province                  0
autonomous_community      0
type                      0
crest_elevation         183
dam_height              183
report                    0
dtype: int64

#### Riverbed Missing Values

Sin nombre means without name, those are NaNs, as we said at detailed_reservoirs.ipynb from the EDA folder

In [16]:
mask = detailed_reservoirs_pd['riverbed'] == 'sin nombre'
detailed_reservoirs_pd.loc[mask, 'riverbed'] = np.nan

In [17]:
def impute_nearest_neighbour(detailed_reservoirs_data, column_name):
    nan_reservoirs = detailed_reservoirs_data[detailed_reservoirs_data[column_name].isna()]
    non_nan_reservoirs = detailed_reservoirs_data[detailed_reservoirs_data[column_name].notna()]

    nan_coord_rads = np.radians(nan_reservoirs[['longitude', 'latitude']].to_numpy()).reshape(-1, 2)
    non_nan_coord_rads = np.radians(non_nan_reservoirs[['longitude', 'latitude']].to_numpy()).reshape(-1, 2)

    distances = haversine_distances(nan_coord_rads, non_nan_coord_rads) * 6371.0 
    closest_indices = distances.argmin(axis=1)

    # Create a mapping from indices to reservoir values
    reservoir_mapping = non_nan_reservoirs[column_name].iloc[closest_indices]
    detailed_reservoirs_data.loc[nan_reservoirs.index, column_name] = reservoir_mapping.values
    return detailed_reservoirs_data

In [18]:
detailed_reservoirs_pd = impute_nearest_neighbour(detailed_reservoirs_pd, 'riverbed')

### Crest Elevation Missing Values

In [19]:
detailed_reservoirs_pd.isna().sum()

name                      0
longitude                 0
latitude                  0
basin                     0
riverbed                  0
google                  252
openstreetmap           331
wikidata                180
province                  0
autonomous_community      0
type                      0
crest_elevation         183
dam_height              183
report                    0
dtype: int64

In [20]:
detailed_reservoirs_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   name                  367 non-null    object 
 1   longitude             367 non-null    float64
 2   latitude              367 non-null    float64
 3   basin                 367 non-null    object 
 4   riverbed              367 non-null    object 
 5   google                115 non-null    object 
 6   openstreetmap         36 non-null     object 
 7   wikidata              187 non-null    object 
 8   province              367 non-null    object 
 9   autonomous_community  367 non-null    object 
 10  type                  367 non-null    object 
 11  crest_elevation       184 non-null    object 
 12  dam_height            184 non-null    object 
 13  report                367 non-null    object 
dtypes: float64(2), object(12)
memory usage: 40.3+ KB


In [21]:
detailed_reservoirs_pd = impute_nearest_neighbour(detailed_reservoirs_pd, 'crest_elevation')

In [22]:
# Now that we have the crest elevation filled, we can convert it to float
detailed_reservoirs_pd['crest_elevation'] = detailed_reservoirs_pd['crest_elevation'].astype(float)

### Removing duplicated rows

In [23]:
# First sight:
mask_duplicated = detailed_reservoirs_pd['name'].value_counts() > 1
duplicated = detailed_reservoirs_pd.loc[detailed_reservoirs_pd['name'].map(mask_duplicated)].sort_values('name')
duplicated.head(10)

,name,longitude,latitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
338,aguilar campoo,42.789079,-4.291817,duero,rio pisuerga,NaN,NaN,NaN,palencia,castilla y leon,presa materiales sueltos homogenea,840.0,NaN,https://sig.mapama.gob.es/WebServices/clientew...
337,aguilar campoo,42.786754,-4.295204,duero,rio pisuerga,NaN,NaN,NaN,palencia,castilla y leon,presa materiales sueltos homogenea,840.0,NaN,https://sig.mapama.gob.es/WebServices/clientew...
336,aguilar campoo,42.795091,-4.287344,duero,rio pisuerga,NaN,NaN,NaN,palencia,castilla y leon,presa fabrica gravedad (hormigon vibrado),840.0,NaN,https://sig.mapama.gob.es/WebServices/clientew...
102,alsa torina,43.094733,-3.999959,cantabrico occidental,rio torina o torino,NaN,NaN,NaN,cantabria,cantabria,presa fabrica arco gravedad,840.0,NaN,https://sig.mapama.gob.es/WebServices/clientew...
101,alsa torina,43.072080,-4.012893,cantabrico occidental,arroyo mojon,NaN,NaN,NaN,cantabria,cantabria,presa materiales sueltos pantalla asfaltica,840.0,NaN,https://sig.mapama.gob.es/WebServices/clientew...
186,anarbe,43.212302,-1.875245,cantabrico oriental,anarbe ibaia o enobietaku erreka,NaN,NaN,NaN,gipuzkoa guipuzcoa,pais vasco,presa fabrica arco gravedad,163.5,79.5,https://sig.mapama.gob.es/WebServices/clientew...
187,anarbe,43.212673,-1.878279,cantabrico oriental,anarbe ibaia o enobietaku erreka,NaN,NaN,NaN,gipuzkoa guipuzcoa,pais vasco,presa fabrica gravedad (hormigon vibrado),162.5,7.5,https://sig.mapama.gob.es/WebServices/clientew...
53,arcos,36.751339,-5.795583,guadalete y barbate,rio guadalete,NaN,NaN,https://www.wikidata.org/wiki/Q17279790,cadiz,andalucia,presa fabrica gravedad (hormigon vibrado),108.5,NaN,https://sig.mapama.gob.es/WebServices/clientew...
54,arcos,36.751390,-5.791509,guadalete y barbate,rio guadalete,NaN,NaN,https://www.wikidata.org/wiki/Q17279790,cadiz,andalucia,presa materiales sueltos zonificada o nucleo,108.5,NaN,https://sig.mapama.gob.es/WebServices/clientew...
255,arenos,40.093352,-0.544595,jucar,barranc jau,NaN,NaN,https://www.wikidata.org/wiki/Q3376336,castello castellon,comunitat valenciana,presa materiales sueltos pantalla material sin...,603.0,18.0,https://sig.mapama.gob.es/WebServices/clientew...


In [24]:
# We well keep the first one that has the highest dam height
detailed_reservoirs_pd = detailed_reservoirs_pd.sort_values('dam_height', ascending=False).drop_duplicates('name')

### Dam height Column

- Missing values have been studied at EDA/detailed_reservoir.ipynb and it was decided to leave the missing values as they were.
- Several types of Random Forest Regressors were tried to impute missing values in the dam height column, but the predictions turned out to be random.

In [28]:
# We convert non-nan dam height to float
mask = detailed_reservoirs_pd['dam_height'].isna() == False
detailed_reservoirs_pd.loc[mask, 'dam_height'] = detailed_reservoirs_pd.loc[mask, 'dam_height'].replace(',', '.').astype(float)

In [30]:
detailed_reservoirs_pd.info()

<class 'pandas.core.frame.DataFrame'>
Index: 320 entries, 355 to 366
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   name                  320 non-null    object 
 1   longitude             320 non-null    float64
 2   latitude              320 non-null    float64
 3   basin                 320 non-null    object 
 4   riverbed              320 non-null    object 
 5   google                100 non-null    object 
 6   openstreetmap         29 non-null     object 
 7   wikidata              162 non-null    object 
 8   province              320 non-null    object 
 9   autonomous_community  320 non-null    object 
 10  type                  320 non-null    object 
 11  crest_elevation       320 non-null    float64
 12  dam_height            159 non-null    object 
 13  report                320 non-null    object 
dtypes: float64(3), object(11)
memory usage: 37.5+ KB


In [31]:
cleaned_detailed_reservoirs_path = PATHS['cleaned_data_notebooks'] / 'detailed_reservoirs_cleaned.csv'
cleaned_detailed_reservoirs_path.parent.mkdir(parents=True, exist_ok=True)
detailed_reservoirs_pd.to_csv(cleaned_detailed_reservoirs_path, index=False)